## CTGF calcium imaging_Using z-scores to: classify response types, calculate proportions, find peak-through, determine nAUC
By Debora Masini, 2024 and 2025

This notebook is used to characterize how individual neurons responded during each behavior (non-responsive, activated, inhibited). It is devided in 2 steps.

  - Step 1 runs classification and plots the pie charts. Step 2 plots line graphs, extracts multiple descriptive values and run Area of the curve.

### Step 1) Performs behavior-specific classification of single-cell activity based on calcium signal traces. 

For each summary_Z-score*.csv file (one per behavior), it identifies the time column and all cell columns, then computes baseline statistics within a fixed baseline window (10–40, blue shade in line graphs). For each cell, it compares the mean activity during a behavior-specific window (provided via BEHAVIOR_WINDOWS, grey shade in line graphs) to its baseline mean. A dynamic threshold is applied, defined as k × baseline SD (default k=2.0). Cells are classified as during-activated if the mean increase exceeds +2 SD, during-inhibited if it decreases below −2 SD, and non-responder otherwise.

For each behavior, the script generates:
 - A categorical summary table with counts and percentages per response class.
 - A pie chart visualization (fixed order and color scheme).
 - A CSV export of the classification results and a cross-behavior consistency table at the cell_id level, marking whether each cell retains the same response category across all behaviors.


In [ ]:
from __future__ import annotations
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, Mapping

In [ ]:
# The directory 
ANALYSIS_DIR = Path(r"...\python_analysis_results\results\collected_summary_files")

# Reading BEHAVIOR_WINDOWS, to be marked in grey on line plots for reader acces.
WINDOWS_FILE = ANALYSIS_DIR / "behavior_windows.csv"
df_windows = pd.read_csv(WINDOWS_FILE)

# Note: the zero of the peri-stimulus plot corresponds to "time"=70.
# windown defined per behavior duration, start and end position+SEM, small adjustment for behaviors with simmilar window size, plot as grey for visual inspection in step 2
BEHAVIOR_WINDOWS = {
    row["behavior"]: (int(row["start"]), int(row["end"]))
    for _, row in df_windows.iterrows()
}

BASELINE_WINDOW = (10, 40) # same for all behaviors, secons -5 to -2 prior to the zero dashed line



# basic read functions
def _detect_time_column(df):
    for c in df.columns:
        if "time" in str(c).lower():
            return c
    raise ValueError("No time column found.")

def _cell_columns(df):
    excluded = {"day", "behavior", "fictive time"}
    return [c for c in df.columns
            if c.strip().startswith("C") and c.lower() not in excluded]

def _window_mask(df, time_col, window): # The script only uses the values in the time column, never the row number. this is safe :)
    start, end = window
    return df[time_col].between(start, end)

# !Methods 1 and 2 did not correctly process the data (we are working with raw trace, not the normalized), I took them away for simplicity.
# METHOD3: is a dynamic threshold based on a change equivalent to 2 SD for a given cell vs its baseline window, used in article.
def classify_activation_state_dynamic(
    df,
    baseline_window = BASELINE_WINDOW,
    behavior_window = BEHAVIOR_WINDOWS,
    k=2.0, # 2 standard deviations
):
    time_col = _detect_time_column(df)

    base_mask = _window_mask(df, time_col, baseline_window)
    beh_mask  = _window_mask(df, time_col, behavior_window)

    out = {}

    for col in _cell_columns(df):
        base_vals = df.loc[base_mask, col]
        beh_vals  = df.loc[beh_mask, col]

        base_mean = base_vals.mean()
        beh_mean  = beh_vals.mean()
        base_std  = base_vals.std()

        if any(pd.isna([base_mean, beh_mean, base_std])):
            continue

        threshold = k * base_std
        delta = beh_mean - base_mean

        if delta >= threshold:
            out[col] = "during-activated"
        elif delta <= -threshold:
            out[col] = "during-inhibited"
        else:
            out[col] = "non-responder"
    
    return out


def summarize_responses(responses, categories=None):
    if categories is None:
        cats = sorted(set(responses.values()))
    else:
        cats = list(categories)

    count = {c: 0 for c in cats}
    for v in responses.values():
        if v in count:
            count[v] += 1

    total = sum(count.values()) or 1
    return pd.DataFrame({"category": cats, "count":   [count[c] for c in cats], "percent": [100 * count[c] / total for c in cats],})


def build_cross_behavior_table(method3_results):
    rows = []
    all_beh = sorted(method3_results)

    for beh in all_beh:
        r = method3_results[beh]
        for cid in sorted(r):
            rows.append({
                "behavior": beh,
                "cell_id": cid,
                "method3_category": r[cid],
            })

    df = pd.DataFrame(rows)

    grp = df.groupby("cell_id")["method3_category"]
    df["method3_consistent"] = grp.transform(
        lambda v: len(set(v) - {""}) <= 1
    )

    return df
    # Interpretation of resulting "method3_consistent":
        # Since the consistency check is computed at the cell_id level, every row belonging to a given cell_id receives the same Boolean.
        # True → This cell_id never changes its method3 category across behaviors.
        # False → This cell_id receives different method3 categories depending on behavior....

def _plot_pie_safe(percent, labels, ax, title="", n_cells=None): # plot pizzas :)
    import pandas as pd

    desired_order = ["non-responder", "during-activated", "during-inhibited"] #plot order
    # Fixed colors
    color_map = {
        "non-responder": "grey",
        "during-activated": "orange",
        "during-inhibited": "darkblue",
    }


    full_title = f"{title}  (n={n_cells})" if n_cells is not None else title # Title  
    percent = pd.Series(percent.values, index=labels) # Convert to Series for safe reindexing
    # Reorder
    percent = percent.reindex(desired_order)
    labels = percent.index.tolist()
    colors = [color_map[l] for l in labels] # Colors in correct order...
    # Validate
    if (
        percent.isna().all()
        or (percent.fillna(0) == 0).all()
        or percent.fillna(0).sum() == 0
    ):
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center")
        ax.set_title(full_title)
        ax.axis("off")
        return
    # Pie chart with fixed start at 12 o’clock
    ax.pie( percent, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90, wedgeprops={"alpha": 0.6} )
    ax.set_title(full_title)
    ax.axis("equal")


#  prep run, main analysis
def analyze_collected_files(
    directory: Path = ANALYSIS_DIR,
    baseline_window = BASELINE_WINDOW,
    behavior_windows = BEHAVIOR_WINDOWS,
    k=2.0, # 2 standard deviations
):
    summary_files = list(directory.glob("summary_Z-score*.csv"))
    if not summary_files:
        raise FileNotFoundError("No summary_Z-score CSVs inside analysis directory.")

    if behavior_windows is None:
        raise ValueError("behavior_windows mapping must be provided.")
    
    method3 = {}
    method3_tables = {}

    for f in summary_files:
        df = pd.read_csv(f)

        beh = (
            f.stem.replace("summary_Z-score", "")
            .strip()
            .replace("_", " ")
        )
        
        if beh not in behavior_windows: # fetch behavior-specific window
            raise KeyError(f"No behavior_window provided for behavior: '{beh}'")

        behavior_window = behavior_windows[beh]
        
        # Method 3 classification
        r3 = classify_activation_state_dynamic(
            df,
            baseline_window=baseline_window,
            behavior_window=behavior_window,
            k=k,
        )
        method3[beh] = r3

        t3 = summarize_responses(r3,categories=["during-activated", "during-inhibited", "non-responder"])
        method3_tables[beh] = t3
        t3.to_csv(directory / f"CLASS_{f.stem}.csv", index=False)

        fig, ax = plt.subplots()
        _plot_pie_safe(
            t3["percent"],
            t3["category"],
            ax,
            title=f"CLASS — {beh}",
            n_cells=len(r3)
        )
        fig.savefig(directory / f"CLASS_{f.stem}_pizza.svg") # pizza graphs
        plt.close(fig)

    cross = build_cross_behavior_table(method3)
    cross.to_csv(directory / "cross_behavior_consistency.csv", index=False)
    return {"method3_summaries": method3_tables, "cross_behavior": cross}

# run
results = analyze_collected_files(
    baseline_window = BASELINE_WINDOW,
    behavior_windows = BEHAVIOR_WINDOWS, # passing behav windows to the function
    k=2.0,
)

In [ ]:
# Step 1 ends, clear all variables
%reset -f
print("memory clean!")

# ---------------------------------------------------
### Step 2) Generate per-behavior Z-score traces grouped by neuron classification with baseline-normalized signals. 

This segment implements a behavior-wise analysis pipeline that baseline-normalizes single-cell Z-score traces, classifies cellular responses using a dynamic SD-based threshold, and generates class-level temporal summaries. For each behavior-specific summary file, traces are first shifted so that the mean activity within a fixed baseline window is zero. Cells are then classified as during-activated, during-inhibited, or non-responder based on whether the change in mean activity during the behavior window exceeds ± k times the baseline standard deviation (default k = 2.0). For each response class, mean and SEM traces are computed and visualized over time, with baseline and behavior windows highlighted. In addition, peak and trough times (and corresponding values) are extracted per cell, saved to CSV, and summarized in bar plots showing mean ± SEM with individual cell scatter overlay. The procedure repeats to all behavior files.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Mapping, MutableMapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### 2a) Classify and generate the class traces as line graphs, normalized to the baseline window z score.

In [ ]:
#Step 2a)
ANALYSIS_DIR = Path(r"...\python_analysis_results\results\collected_summary_files")

WINDOWS_FILE = ANALYSIS_DIR / "behavior_windows.csv"
df_windows = pd.read_csv(WINDOWS_FILE)
BEHAVIOR_WINDOWS = {
    row["behavior"]: (int(row["start"]), int(row["end"]))
    for _, row in df_windows.iterrows()
}

BASELINE_WINDOW = (10, 40) # same for all behaviors

CLASS_COLORS = {
    "during-activated": "orange",
    "during-inhibited": "darkblue",
    "non-responder": "grey",
}
CLASS_ORDER = list(CLASS_COLORS)


def _behavior_from_filename(path: Path) -> str:
    return path.stem.replace("summary_Z-score", "").strip().replace("_", " ")

def _detect_time_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        if "time" in str(col).lower():
            return col
    raise ValueError("No time column found in provided DataFrame.")


def _cell_columns(df: pd.DataFrame) -> list[str]:
    excluded = {"day", "behavior", "fictive time"}
    return [
        col
        for col in df.columns
        if col.strip().startswith("C") and col.lower() not in excluded
    ]


def _window_mask(df: pd.DataFrame, time_col: str, window: Sequence[float]) -> pd.Series:
    start, end = window
    return df[time_col].between(start, end)

# baseline mean-centering of z-score trace
def normalize_to_baseline(
    df: pd.DataFrame, *, baseline_window: Sequence[float], time_col: str
) -> pd.DataFrame:

    base_mask = _window_mask(df, time_col, baseline_window)
    normalized = df.copy()

    for col in _cell_columns(df):
        base_mean = df.loc[base_mask, col].mean()
        if pd.isna(base_mean):
            continue
        normalized[col] = df[col] - base_mean # data already z-scored, otherwise... vs normalized[col] = (df[col] - base_mean) / base_std

    return normalized

# same as step 1
def classify_activation_state_dynamic(
    df: pd.DataFrame,
    *,
    baseline_window: Sequence[float] = BASELINE_WINDOW,
    behavior_window: Sequence[float] = None, # must pass on the caller or =bug
    k: float = 2.0,
) -> Dict[str, str]:

    time_col = _detect_time_column(df)
    base_mask = _window_mask(df, time_col, baseline_window)
    beh_mask = _window_mask(df, time_col, behavior_window)

    out: Dict[str, str] = {}

    for col in _cell_columns(df):
        base_vals = df.loc[base_mask, col]
        beh_vals = df.loc[beh_mask, col]

        base_mean = base_vals.mean()
        beh_mean = beh_vals.mean()
        base_std = base_vals.std()

        if any(pd.isna([base_mean, beh_mean, base_std])):
            continue

        threshold = k * base_std
        delta = beh_mean - base_mean

        if delta >= threshold:
            out[col] = "during-activated"
        elif delta <= -threshold:
            out[col] = "during-inhibited"
        else:
            out[col] = "non-responder"

    return out

# Return mean and SEM traces for each classification class.
def _compute_class_traces(
    df: pd.DataFrame, classifications: Mapping[str, str], time_col: str
) -> Dict[str, tuple[pd.Series, pd.Series]]:

    traces: Dict[str, tuple[pd.Series, pd.Series]] = {}
    df_time_indexed = df.set_index(time_col)

    for cls in CLASS_ORDER:
        cells = [cid for cid, c in classifications.items() if c == cls]
        if not cells:
            continue

        subset = df_time_indexed[cells]
        mean = subset.mean(axis=1)
        sem = subset.sem(axis=1)
        traces[cls] = (mean, sem)

    return traces

# Return per-cell peak and trough positions (time + value)>> Note: finds global peak/trough over the whole peri-stimulus period.
def _peak_trough_positions(
    df: pd.DataFrame, classifications: Mapping[str, str], time_col: str
) -> pd.DataFrame:

    df_time_indexed = df.set_index(time_col)
    records = []

    for cell_id, cls in classifications.items():
        if cell_id not in df_time_indexed:
            continue

        series = df_time_indexed[cell_id]
        if series.empty:
            continue

        peak_time = series.idxmax()
        trough_time = series.idxmin()

        records.append(
            {
                "cell_id": cell_id,
                "class": cls,
                "peak_time": peak_time,
                "peak_value": series.loc[peak_time],
                "trough_time": trough_time,
                "trough_value": series.loc[trough_time],
            }
        )

    return pd.DataFrame.from_records(records)


# Generate class-level trace plot and peak/trough summaries for a behavior. LINE GRAPHS
def plot_class_traces_for_behavior(
    file_path: Path,
    *,
    baseline_window: Sequence[float],
    behavior_windows: Mapping[str, Sequence[float]],
    k: float = 2.0,
    output_dir: Path | None = None,
) -> Dict[str, Path | None]:

    df = pd.read_csv(file_path)
    behavior = _behavior_from_filename(file_path)

    if behavior not in behavior_windows:
        raise KeyError(f"No behavior window configured for '{behavior}'")

    behavior_window = behavior_windows[behavior]
    time_col = _detect_time_column(df)

    normalized_df = normalize_to_baseline(
        df, baseline_window=baseline_window, time_col=time_col
    )

    classifications = classify_activation_state_dynamic(
        normalized_df,
        baseline_window=baseline_window,
        behavior_window=behavior_window,
        k=k,
    )
    traces = _compute_class_traces(normalized_df, classifications, time_col)

    if not traces:
        return {"trace_plot": None, "position_plot": None, "position_csv": None}

    fig, ax = plt.subplots(figsize=(10, 6))
    for cls in CLASS_ORDER:
        if cls not in traces:
            continue

        mean, sem = traces[cls]
        color = CLASS_COLORS[cls]
        ax.plot(mean.index, mean.values, label=cls, color=color)
        ax.fill_between(
            mean.index, mean - sem, mean + sem, color=color, alpha=0.2
        )

    ax.axvspan(*baseline_window, color="lightblue", alpha=0.15, label="baseline")
    ax.axvspan(*behavior_window, color="lightgrey", alpha=0.2, label="behavior")

    ax.set_xlabel("Fictive Time (mean ± SEM)")
    ax.set_ylabel("Baseline-normalized Z-score")
    ax.set_title(f"Z-score over time (baseline-normalized) — {behavior}")
    ax.legend()
    fig.tight_layout()

    out_dir = output_dir or file_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    trace_path = out_dir / f"CLASS_TRACE_BASELINE_NORM_{file_path.stem}.svg" # LINE PLOTS
    fig.savefig(trace_path, dpi=300)
    plt.close(fig)

    position_df = _peak_trough_positions(normalized_df, classifications, time_col)
    position_df.insert(0, "behavior", behavior)
    csv_path = out_dir / f"PEAK_TROUGH_POSITIONS_{file_path.stem}.csv" # data for plotting elsewhere
    position_df.to_csv(csv_path, index=False)

    return {"trace_plot": trace_path, "position_csv": csv_path,}

# run final prep
def plot_all_behaviors(
    directory: Path = ANALYSIS_DIR,
    *,
    baseline_window: Sequence[float] = BASELINE_WINDOW,
    behavior_windows: Mapping[str, Sequence[float]] = BEHAVIOR_WINDOWS,
    k: float = 2.0,
    output_dir: Path | None = None,
) -> MutableMapping[str, Dict[str, Path | None]]:

    outputs: MutableMapping[str, Dict[str, Path | None]] = {}

    for file_path in sorted(directory.glob("summary_Z-score*.csv")):
        behavior = _behavior_from_filename(file_path)
        out_path = plot_class_traces_for_behavior(
            file_path,
            baseline_window=baseline_window,
            behavior_windows=behavior_windows,
            k=k,
            output_dir=output_dir,
        )
        outputs[behavior] = out_path

    return outputs

# run
if __name__ == "__main__":
    results = plot_all_behaviors()
    for behavior, artifacts in results.items():
        trace_msg = ( f"trace plot: {artifacts['trace_plot'].name}" if artifacts["trace_plot"] else "trace plot: none")
        csv_msg = ( f"csv: {artifacts['position_csv'].name}" if artifacts["position_csv"] else "csv: none")
        print(f"{behavior}: {trace_msg} | {csv_msg}")

### Step 2b) generate csv files of data on plots: computes class-level mean, SD, SEM, 95% CI, and sample counts over time, also writes a long-format CSV that can be used to recreate the plots in other software packages.

In [ ]:
# Step 2b)

# Return mean and SEM traces for each classification class. >>> Slight update to include more stats info! now as dict, not a tuple.
def _compute_class_traces(
    df: pd.DataFrame, classifications: Mapping[str, str], time_col: str
) -> Dict[str, Dict[str, pd.Series]]:

    traces: Dict[str, Dict[str, pd.Series]] = {}
    df_time_indexed = df.set_index(time_col)

    for cls in CLASS_ORDER:
        cells = [cid for cid, c in classifications.items() if c == cls]
        if not cells:
            continue

        subset = df_time_indexed[cells]
        mean = subset.mean(axis=1)
        sem = subset.sem(axis=1)
        sd = subset.std(axis=1, ddof=1)
        n = subset.notna().sum(axis=1)
        ci95 = 1.96 * sem
        traces[cls] = {"mean": mean, "sem": sem, "sd": sd, "ci95": ci95, "n": n} # added info and n for checks

    return traces

# stats into a long-format dataframe for export.
def _traces_to_long_dataframe(
    traces: Mapping[str, Dict[str, pd.Series]], *, behavior: str, time_label: str
) -> pd.DataFrame:

    rows = []
    for cls, stats in traces.items():
        rows.append(
            pd.DataFrame(
                {
                    "behavior": behavior,
                    "class": cls,
                    time_label: stats["mean"].index,
                    "mean": stats["mean"].values,
                    "sem": stats["sem"].values,
                    "sd": stats["sd"].values,
                    "ci95": stats["ci95"].values,
                    "n": stats["n"].values,
                }
            )
        )

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["behavior", "class", time_label, "mean", "sem", "sd", "ci95", "n"]
    )


# we go again, generate a CSV of baseline-normalized class trace stats for a behavior.
def export_baseline_norm_stats_for_behavior(
    file_path: Path,
    *,
    baseline_window: Sequence[float],
    behavior_windows: Mapping[str, Sequence[float]],
    k: float = 2.0,
    output_dir: Path | None = None,
) -> Path | None:
    df = pd.read_csv(file_path)
    behavior = _behavior_from_filename(file_path)

    if behavior not in behavior_windows:
        raise KeyError(f"No behavior window configured for '{behavior}'")

    behavior_window = behavior_windows[behavior]
    time_col = _detect_time_column(df)

    normalized_df = normalize_to_baseline(
        df, baseline_window=baseline_window, time_col=time_col
    )

    classifications = classify_activation_state_dynamic(
        normalized_df,
        baseline_window=baseline_window,
        behavior_window=behavior_window,
        k=k,
    )
    traces = _compute_class_traces(normalized_df, classifications, time_col)

    if not traces:
        return None

    stats_df = _traces_to_long_dataframe(traces, behavior=behavior, time_label=time_col)

    out_dir = output_dir or file_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"CLASS_TRACE_NORM_{file_path.stem}.csv" ##### to be used on step 2c
    stats_df.to_csv(out_path, index=False)

    return out_path

# prep for run: Loop over every behavior and export baseline-normalized trace stats.
def export_all_baseline_norm_stats(
    directory: Path = ANALYSIS_DIR,
    *,
    baseline_window: Sequence[float] = BASELINE_WINDOW,
    behavior_windows: Mapping[str, Sequence[float]] = BEHAVIOR_WINDOWS,
    k: float = 2.0,
    output_dir: Path | None = None,
) -> MutableMapping[str, Path | None]:

    outputs: MutableMapping[str, Path | None] = {}

    for file_path in sorted(directory.glob("summary_Z-score*.csv")):
        behavior = _behavior_from_filename(file_path)
        try:
            outputs[behavior] = export_baseline_norm_stats_for_behavior(
                file_path,
                baseline_window=baseline_window,
                behavior_windows=behavior_windows,
                k=k,
                output_dir=output_dir,
            )
        except KeyError:
            outputs[behavior] = None

    return outputs

# run
if __name__ == "__main__":
    results = export_all_baseline_norm_stats()
    for behavior, csv_path in results.items():
        msg = csv_path.name if csv_path else "no stats generated"
        print(f"{behavior}: {msg}")

### step 2c) the normalized area (area of the pool)

In [ ]:
# Step2c) 
STATS_PREFIX = "CLASS_TRACE_NORM_"

# creates a boolean mask for filtering a time vector, selecting times within [start, end]
def _behavior_mask(times: pd.Series, behavior_window: Sequence[float]) -> pd.Series:
    start, end = behavior_window
    return (times >= start) & (times <= end)


# Infer behavior name from a stats filename
def _behavior_from_stats_filename(stats_path: Path) -> str:

    stem = stats_path.stem
    if stem.startswith(STATS_PREFIX):
        stem = stem[len(STATS_PREFIX):]

    # Prefer splitting on the common 'summary_Z-score' token
    token = "summary_Z-score"
    if token in stem:
        after = stem.split(token, 1)[1]
        after = after.lstrip("_- ")
        return after or stem

    return stem # Otherwise return remainder

# Return (normalized_area, raw_area) for a given class in behavior window. attention: NumPy historically uses np.trapz. np.trapezoid exists only in newer versions.
def _summarize_class_area(
    df: pd.DataFrame, *, cls: str, time_col: str, behavior_window: Sequence[float]
) -> Tuple[float, float]:
    subset = df[df["class"] == cls]
    if subset.empty:
        raise ValueError(f"no rows for class '{cls}'")

    subset = subset.sort_values(time_col)
    window_mask = _behavior_mask(subset[time_col], behavior_window)
    window_slice = subset.loc[window_mask]

    if window_slice.empty:
        raise ValueError(f"no data for class '{cls}' in behavior window {behavior_window}")

    times = window_slice[time_col].to_numpy()
    values = window_slice["mean"].to_numpy()

    if cls == "during-inhibited":
        raw_area = np.trapezoid(-values, times)  # pool area: forces positive area for inhibited class by negating mean trace.
    else:
        raw_area = np.trapezoid(values, times)

    duration = float(behavior_window[1] - behavior_window[0])
    normalized_area = raw_area / duration if duration else np.nan
    return float(normalized_area), float(raw_area)

# Print normalized area summaries for a single stats CSV.
def summarize_area_from_stats(stats_path: Path, *, behavior_windows: Mapping[str, Sequence[float]]) -> None:
    behavior = _behavior_from_stats_filename(stats_path)
    if behavior not in behavior_windows:
        print(f"{behavior}: behavior window not configured; skipping")
        return

    df = pd.read_csv(stats_path)
    if df.empty:
        print(f"{behavior}: stats file is empty; skipping")
        return

    time_col = _detect_time_column(df)
    behavior_window = behavior_windows[behavior]

    #print(f"\n{behavior} (behavior window {behavior_window[0]}-{behavior_window[1]} {time_col})")

    present_classes = set(df["class"].unique())
    for cls in CLASS_ORDER:
        if cls not in present_classes:
            continue
        try:
            normalized_area, raw_area = _summarize_class_area(
                df, cls=cls, time_col=time_col, behavior_window=behavior_window
            )
        except ValueError as exc:
            print(f"  {cls}: {exc}")
            continue
        print(f"  {cls}: normalized area={normalized_area:.4f} (raw area={raw_area:.4f})")

# run
def summarize_all_normalized_areas(
    directory: Path = ANALYSIS_DIR, *, behavior_windows: Mapping[str, Sequence[float]] = BEHAVIOR_WINDOWS
) -> None:

    pattern = f"{STATS_PREFIX}summary_Z-score*.csv"
    paths = sorted(directory.glob(pattern))
    if not paths:
        print(f"No files matched: {directory / pattern}")
        return

    for stats_path in paths:
        summarize_area_from_stats(stats_path, behavior_windows=behavior_windows)

summarize_all_normalized_areas()